# 🚰 Week 7 · The hostel agent: it doesn't just answer, it *acts*

11:40 pm. A student messages the hostel helpdesk: water is pouring from a pipe in the C-block bathroom.
Last week our app could **answer** questions. Tonight it has to **do things**: find the warden's number, call people, get the water shut off, check it worked, and keep everyone informed.

Nobody gives it the steps. It gets a **goal** and some **tools**, and it works out the plan one step at a time: **think → act → observe → repeat**. That loop is what makes it an **agent**.

**How to use this notebook**
- Run each cell with ▶ (or **Shift + Enter**), top to bottom.
- Cells marked **✏️ YOUR TURN** are where you change one line and run again.
- Stuck? Type the **section number** in the Zoom chat.

**Before you start:** File → **Save a copy in Drive**. Same `GEMINI_API_KEY` as Weeks 5 and 6.

## 🏠 Start here · The problem, and the manual

The complaint is below. The hostel's plumbing manual is a spreadsheet, one row per paragraph: **[open it as a table](https://github.com/san81/ai-course/blob/main/docs/weeks/week7/valve-manual.csv)**. Look for a minute:

- 🎨 **The levers are colour-coded**, and the wrong one can stop water to the whole hostel, the kitchen, or even the fire line.
- 🔄 **It changed:** page 42 (2019) says *yellow* is C-block. Page 44 (revision 2024) adds an *orange* lever just for the 2nd floor.
- 📞 **It has no phone numbers** (page 90): those come from the hostel's contact list.

A person on night duty would never read all of it. They'd use the **index**, go to the right page, and act. So will our agent.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 160)

BASE = "https://raw.githubusercontent.com/san81/ai-course/main/docs/weeks/week7/"
manual = pd.read_csv(BASE + "valve-manual.csv")
manual.to_csv("valve-manual.csv", index=False)          # a local copy for Mr. Chenna Reddy (section 3)

COMPLAINT = ("Anna, water is pouring from the pipe under the sink in the C-block 2nd floor "
             "bathroom!! The floor is flooding and it's going into the corridor 😰")

print("📩 New complaint, 11:40 pm:\n  ", COMPLAINT, "\n")
print(len(manual), "paragraphs in the manual")
manual

✏️ **Your turn (1 minute):** you're on night duty. Who do you contact **first**? Type one letter in the chat: **A** the plumber · **B** the warden · **C** post in the hostel WhatsApp group · **D** go and look yourself.

## 0 · Setup (2 minutes)

Same as Weeks 5 and 6: `GEMINI_API_KEY` in the 🔑 Secrets panel, **Notebook access** on. Run the three cells until you see **✅ Connected**.

In [ ]:
!pip -q install -U google-genai mcp

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:                      # not running in Colab
    import os, getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Paste your Gemini API key: ")
except Exception as error:
    print("🔑 Add GEMINI_API_KEY in the Secrets panel (key icon, left) and switch on 'Notebook access'.")
    raise error

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"   # ✏️ free tier. If the agent gets confused, try "gemini-3.6-flash"
print("Key loaded. Now run the next cell to test it.")

In [ ]:
# 🛟 Turns confusing error messages into plain advice, then checks your key works.
import time, asyncio
from google.genai import errors

def explain_api_error(e):
    code, text = getattr(e, "code", None), str(e)
    if code == 400 and "API key" in text:
        return ("🔑 Your key was not accepted. Copy it again from aistudio.google.com (no spaces), "
                "update the GEMINI_API_KEY secret, and run the setup cells again.")
    if code == 402:
        return ("💳 This key's project is linked to billing with no credit. You do NOT need a card: in "
                "aistudio.google.com create a NEW project (no billing), make a key there, update the secret.")
    if code == 403:
        return "🚫 This Google account can't use the Gemini API. Use a personal Gmail, make a new key, update the secret."
    if code == 404:
        return "🔍 That model isn't available to your key. In the setup cell, change MODEL (e.g. 'gemini-3.6-flash')."
    if code == 429:
        return ("⏳ Free limit reached. Per-minute: wait one minute. Daily: resets at midnight US Pacific "
                "(about 12:30–1:30 pm India time). Meanwhile, follow on the instructor's screen.")
    if code and code >= 500:
        return "🌩️ Google's servers are busy. Wait 30 seconds and run the cell again."
    return None

def _friendly_errors(shell, etype, evalue, tb, tb_offset=None):
    advice = explain_api_error(evalue)
    if advice:
        print(advice); print(f"\n(Technical detail, if your instructor asks: error {getattr(evalue, 'code', '?')})")
    else:
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)

try:
    get_ipython().set_custom_exc((errors.APIError,), _friendly_errors)
except NameError:
    pass

async def with_retry(make_call, tries=3):
    for attempt in range(tries):
        try:
            return await make_call()
        except errors.APIError as e:
            if e.code == 429 and attempt < tries - 1:
                print("   ⏳ rate limit, waiting 30 s…"); await asyncio.sleep(30)
            else:
                raise

try:
    client.models.generate_content(model=MODEL, contents="Reply with just: OK")
    print("✅ Connected. Model:", MODEL)
except errors.APIError as e:
    print(explain_api_error(e) or e)

## 1 · The tools, and you as the agent

An agent is only as capable as its **tools**. Here are the first four: plain Python functions, the same idea as Week 5's `get_block_contact`. The people on the other end are **simulated** (they reply with text, nobody's phone rings), so everyone in class gets the same night.

The `WORLD` settings decide how that night goes. In section 4 you'll change them and watch the agent change its plan.

In [ ]:
import re
from urllib.parse import quote

# 🌍 How tonight goes. Section 4 changes these.
WORLD = {
    "warden_answers": True,            # does the C-block warden pick up?
    "plumber_comes": "12:45 am",       # or e.g. "tomorrow at 9 am"
}
MY_WHATSAPP = ""   # ✏️ optional: your own number like "+91 98xxxxxxxx", so the WhatsApp drafts open to YOU

CONTACTS = {   # the hostel helpdesk's contact list (made-up names and numbers)
    "A": {"warden": ("Mr. Ramesh", "+91 90000 00011"), "deputy warden": ("Mr. Kiran", "+91 90000 00012")},
    "B": {"warden": ("Mrs. Sujatha", "+91 90000 00021"), "deputy warden": ("Mr. Anil", "+91 90000 00022")},
    "C": {"warden": ("Mrs. Lakshmi", "+91 90000 00001"), "deputy warden": ("Mr. Prasad", "+91 90000 00003")},
}
PLUMBER = ("Ravi (plumber)", "+91 90000 00002")
OUTBOX = []    # WhatsApp drafts the agent prepares

def get_block_contact(block: str) -> dict:
    """Look up the warden and deputy warden for a hostel block (A, B or C): names and phone numbers."""
    b = block.strip().upper()[:1]
    if b not in CONTACTS:
        return {"error": f"No block {block}. Blocks are A, B and C."}
    return {role: {"name": n, "phone": p} for role, (n, p) in CONTACTS[b].items()}

def call_person(phone: str, message: str) -> str:
    """Phone someone and tell them what's happening. Returns what they say back."""
    digits = re.sub(r"\D", "", phone)[-10:]
    who = {re.sub(r"\D", "", p)[-10:]: (n, role) for blk in CONTACTS.values() for role, (n, p) in blk.items()}
    if digits == re.sub(r"\D", "", PLUMBER[1])[-10:]:
        if "tomorrow" in WORLD["plumber_comes"]:
            return (f"Ravi here. Sorry, I'm out of town tonight. I can come {WORLD['plumber_comes']} and replace that pipe. "
                    "Tonight, stop the water so it doesn't flood. Mr. Chenna Reddy at the hostel knows the levers.")
        return (f"Ravi here. I'll come at {WORLD['plumber_comes']} and replace the pipe. Meanwhile stop the water "
                "so the corridor doesn't flood. Mr. Chenna Reddy at the hostel knows the levers. Don't open the pipe yourself.")
    if digits not in who:
        return "📵 Wrong number. Nobody by that number."
    name, role = who[digits]
    if role == "warden" and not WORLD["warden_answers"]:
        return f"📵 No answer from {name}. (It's late; the phone rang out.)"
    if "red" in message.lower():
        return (f"{name}: Red stops water to the WHOLE hostel and the kitchen. Only if no smaller lever works. "
                "If it really comes to that, yes, you have my approval.")
    return (f"{name}: Thanks for calling. The plumber is Ravi, {PLUMBER[1]}. Mr. Chenna Reddy, our Master of "
            "Control Panels, can shut the water at the valve junction. Send me a WhatsApp update when it's under control.")

def send_whatsapp(phone: str, message: str) -> str:
    """Prepare a WhatsApp message. A human checks it and presses Send."""
    to = re.sub(r"\D", "", MY_WHATSAPP or phone)
    link = f"https://wa.me/{to}?text={quote(message)}"
    OUTBOX.append({"to": phone, "message": message, "link": link})
    return f"Draft ready for {phone}. A human must open it and press Send: {link}"

def reply_to_student(message: str) -> str:
    """Send the final reply to the student who complained. Call this last."""
    return "✅ Reply delivered to the student."

LOCAL_TOOLS = [get_block_contact, call_person, send_whatsapp, reply_to_student]
print("🧰 Tools:", ", ".join(f.__name__ for f in LOCAL_TOOLS))

In [ ]:
# ✏️ YOUR TURN: be the agent yourself. Run this, read the reply, then call the next person.
print(get_block_contact("C"))
print(call_person("+91 90000 00001", "Water leak in C-block 2nd floor bathroom. What should I do?"))

✏️ The warden gave you a new phone number. Add one line to the cell above that calls it, and run it again. What does that person tell you to do **next**? Type one word in the chat.

You just did what an agent does: **act, read the result, decide the next step.** Now let Gemini do it.

## 2 · The agent loop: think → act → observe

Below is the whole agent. Read the loop in the middle; it's short:

1. Send the goal, the history so far and the list of tools to the model.
2. The model replies with a **plan line** and a **tool request**. It can't run anything itself (Week 5's egg!).
3. **Our code** runs the tool and adds the result to the history.
4. Repeat, until the model stops asking for tools, or we hit a **step limit**.

The model is the judgement. Everything around it (the tool list, running the calls, the history, the step limit) is called the **harness**.

In [ ]:
SYSTEM = """You are the night-duty helper at a college hostel. A student has reported a problem.
GOAL: stop the damage, get it repaired, and keep people informed.
Nobody gives you steps. Decide what to do next, one step at a time, using your tools.
Rules:
- Before each tool call, write ONE short line starting with "Plan:".
- Look up phone numbers with your tools; never invent them.
- Shut the smallest lever that covers the leak. Only shut a bigger one if the smaller one fails,
  and the RED lever only with the warden's approval.
- Before saying the water has stopped, check it yourself.
- Send the warden a WhatsApp update once things are under control.
- Finish with reply_to_student (say what happened and when the plumber comes), then stop."""

MAX_STEPS = 15     # ✏️ the harness's safety limit: an agent must never loop forever
RUNS = {}          # every run's steps, to compare plans in section 4

async def run_agent(complaint, tools=LOCAL_TOOLS, chenna=None, name="run", max_steps=None):
    """The harness: runs the think → act → observe loop and prints every step."""
    max_steps = max_steps or MAX_STEPS
    local = {f.__name__: f for f in tools}
    tool_list, mcp_names = list(tools), set()
    if chenna is not None:                                   # tools offered by an MCP server
        listed = await chenna.list_tools()
        tool_list.append(types.Tool(function_declarations=[
            types.FunctionDeclaration(name=t.name, description=t.description, parameters_json_schema=t.inputSchema)
            for t in listed.tools]))
        mcp_names = {t.name for t in listed.tools}
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM, temperature=0.2, tools=tool_list,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))   # WE run the tools
    history = [types.Content(role="user", parts=[types.Part(text=f"New complaint from a student: {complaint}")])]
    steps = []
    print(f"📩 {complaint}\n")
    for turn in range(1, max_steps + 1):
        response = await with_retry(lambda: client.aio.models.generate_content(
            model=MODEL, contents=history, config=config))
        reply = response.candidates[0].content
        history.append(reply)
        for part in reply.parts or []:
            if part.text and not part.thought:
                print(f"🧠 {part.text.strip()}")
        calls = response.function_calls or []
        if not calls:
            print(f"\n🏁 Done in {turn} turns, {len(steps)} tool calls.")
            break
        results = []
        for call in calls:
            args = dict(call.args or {})
            print(f"🔧 {call.name}({', '.join(f'{k}={v!r}' for k, v in args.items())})")
            if call.name in local:
                result = local[call.name](**args)
            elif call.name in mcp_names:
                out = await chenna.call_tool(call.name, args)
                result = "\n".join(c.text for c in out.content if getattr(c, "text", None))
            else:
                result = f"There is no tool called {call.name}."
            print(f"👀 {str(result)[:400]}\n")
            steps.append(call.name + (f"({args.get('colour')})" if "colour" in args else ""))
            results.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        history.append(types.Content(role="user", parts=results))
        await asyncio.sleep(4)          # stay inside the free per-minute limit
    else:
        print(f"\n⛔ Stopped by the harness: {max_steps} turns used up. (A safety limit, not a crash.)")
    RUNS[name] = steps
    return steps

await run_agent(COMPLAINT, name="1 · no Chenna Reddy")

✏️ **Your turn:** read the 🧠 plan lines. How many **tool calls** did it make? Type **one number**.

Notice what it *couldn't* do. The plumber said "stop the water", but the agent has **no tool for that**. A good agent says so honestly instead of pretending. Next, we give it a helper who can.

## 3 · Mr. Chenna Reddy, the hostel's MCP 🔧

Mr. Chenna Reddy is the hostel's **M**aster of **C**ontrol **P**anels. He can't fix pipes. But he knows the valve junction, has a wrench, and can look things up in the manual.

In AI, **MCP** means **Model Context Protocol**: a standard way for any AI app to ask a helper **"what can you do?"**, get a list of its tools, and then use them. One helper, written once, works with Gemini, Claude, n8n, any app that speaks MCP. Like a USB-C port for AI tools.

The cell below **is** Mr. Chenna Reddy: a real MCP server, about 100 lines. Skim it: four tools, each a Python function with a description. His `look_up_manual` tool is Week 6's keyword search (BM25): the index, then the right page.

In [ ]:
%%writefile chenna_reddy_mcp.py
"""
Mr. Chenna Reddy, the hostel's MCP ("Master of Control Panels"), as a real MCP server.

He can't fix pipes. He CAN look things up in the plumbing manual, shut or open the
coloured levers at the valve junction, and check whether a bathroom is still leaking.

Run it:
  python chenna_reddy_mcp.py           # stdio: an AI app (or our notebook) starts it itself
  python chenna_reddy_mcp.py --http    # http://localhost:8000/mcp  (for n8n or other apps)

Settings (environment variables), so you can change the world and watch the agent adapt:
  MANUAL_CSV     path to the manual (default: valve-manual.csv next to this file)
  LEAK_AT        where the leak is: C-2 (C-block 2nd floor, default), C-1, B-1, A-3 ...
  STUCK_LEVERS   comma-separated levers that won't turn, e.g. "orange"
"""
import csv, math, os, re, sys
from collections import Counter
from mcp.server.fastmcp import FastMCP

HERE = os.path.dirname(os.path.abspath(__file__))
MANUAL_CSV = os.environ.get("MANUAL_CSV", os.path.join(HERE, "valve-manual.csv"))
LEAK_AT = os.environ.get("LEAK_AT", "C-2").upper()
STUCK = {s.strip().lower() for s in os.environ.get("STUCK_LEVERS", "").split(",") if s.strip()}

# Which bathrooms each lever feeds. "C-2" = C-block, 2nd floor.
LEVERS = {
    "red":    {"covers": "EVERY block + kitchen + RO drinking water", "areas": "*"},
    "blue":   {"covers": "A-block, all floors", "areas": "A-"},
    "green":  {"covers": "B-block, all floors", "areas": "B-"},
    "yellow": {"covers": "C-block, all floors (incl. laundry)", "areas": "C-"},
    "orange": {"covers": "C-block 2nd-floor bathrooms only", "areas": "C-2"},
    "white":  {"covers": "RO drinking-water plant only", "areas": "RO"},
}
NEVER = {"black": "the fire hydrant line must always stay open",
         "grey": "that's the kitchen LPG gas wheel, not a water valve"}
shut = set()   # levers currently shut (fresh every time the server starts)

with open(MANUAL_CSV, newline="", encoding="utf-8") as f:
    PAGES = list(csv.DictReader(f))

# --- a tiny BM25 keyword search: "look in the index, then the right page" (Week 6) ---
def _words(t): return re.findall(r"[a-z0-9]+", t.lower())
_DOCS = [_words(p["section"] + " " + p["text"]) for p in PAGES]
_DF = Counter(w for d in _DOCS for w in set(d))
_AVG = sum(map(len, _DOCS)) / len(_DOCS)
def _bm25(query, k1=1.5, b=0.75):
    q, n = _words(query), len(_DOCS)
    scores = []
    for d in _DOCS:
        c, s = Counter(d), 0.0
        for w in q:
            if c[w]:
                idf = math.log((n - _DF[w] + 0.5) / (_DF[w] + 0.5) + 1)
                s += idf * c[w] * (k1 + 1) / (c[w] + k1 * (1 - b + b * len(d) / _AVG))
        scores.append(s)
    return scores

mcp = FastMCP(
    "Mr. Chenna Reddy",
    instructions=("I'm Mr. Chenna Reddy, the hostel's Master of Control Panels. I don't know how to "
                  "fix or replace pipes; call a plumber for that. I can look things up in the plumbing "
                  "manual, use a wrench to shut or open the coloured levers at the valve junction, and "
                  "check whether a bathroom is still leaking."),
)

@mcp.tool()
def look_up_manual(question: str, how_many: int = 3) -> str:
    """Search the hostel plumbing manual (300 pages) and return the most relevant pages.
    Use it to find out which coloured lever controls which area before shutting anything."""
    scores = _bm25(question)
    best = sorted(range(len(PAGES)), key=lambda i: -scores[i])[:max(1, min(how_many, 6))]
    hits = [f"p.{PAGES[i]['page']} ({PAGES[i]['section']}): {PAGES[i]['text']}" for i in best if scores[i] > 0]
    return "\n".join(hits) if hits else "Nothing in the manual matches that. Try other words."

@mcp.tool()
def shut_lever(colour: str, warden_approved: bool = False) -> str:
    """Shut one coloured lever at the valve junction (quarter turn with the wrench).
    colour: red, blue, green, yellow, orange or white.
    The RED lever stops water to the whole hostel; it needs warden_approved=true."""
    c = colour.strip().lower()
    if c in NEVER:
        return f"❌ I won't touch {c}: {NEVER[c]}."
    if c not in LEVERS:
        return f"❌ There's no {c} lever. Levers: {', '.join(LEVERS)}."
    if c == "red" and not warden_approved:
        return ("⚠️ Not yet. RED shuts water to EVERY block, the kitchen and drinking water. "
                "Ask the warden first, then call me again with warden_approved=true.")
    if c in STUCK:
        return f"🔧 The {c} lever is jammed. Even with the 24 mm wrench it won't turn. Pick another lever."
    shut.add(c)
    return f"✅ {c.upper()} lever shut. Water is now off for: {LEVERS[c]['covers']}."

@mcp.tool()
def open_lever(colour: str) -> str:
    """Open a lever again (after the repair)."""
    c = colour.strip().lower()
    if c not in shut:
        return f"The {c} lever is already open."
    shut.discard(c)
    return f"✅ {c.upper()} lever open. Water is back for: {LEVERS[c]['covers']}."

@mcp.tool()
def check_leak(block: str, floor: int) -> str:
    """Go and look: is the bathroom at this block and floor still leaking?"""
    area = f"{block.strip().upper()[:1]}-{floor}"
    if area != LEAK_AT:
        return f"👀 {block.upper()}-block floor {floor}: dry, no leak here."
    stopped = [c for c in shut if LEVERS[c]["areas"] == "*" or area.startswith(LEVERS[c]["areas"])]
    if stopped:
        return f"👀 {area}: the water has stopped (the {', '.join(stopped)} lever is shut). The floor is wet but nothing is dripping."
    return f"👀 {area}: STILL LEAKING. Water is spraying from the pipe joint under the sink."

if __name__ == "__main__":
    if "--http" in sys.argv:
        mcp.settings.host = "0.0.0.0"
        mcp.settings.port = int(os.environ.get("PORT", "8000"))
        mcp.run(transport="streamable-http")      # → http://localhost:8000/mcp
    else:
        mcp.run()                                 # stdio

In [ ]:
import os, sys
from contextlib import asynccontextmanager
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

@asynccontextmanager
async def chenna_reddy(**settings):
    """Start Mr. Chenna Reddy's MCP server and connect to him. settings change his world, e.g. STUCK_LEVERS="orange"."""
    env = {**os.environ, **{k: str(v) for k, v in settings.items()}}
    params = StdioServerParameters(command=sys.executable, args=["chenna_reddy_mcp.py"], env=env)
    async with stdio_client(params, errlog=open("chenna_reddy.log", "w")) as (read, write):
        async with ClientSession(read, write) as session:
            session.hello = await session.initialize()
            yield session

# "Mr. Chenna Reddy, what can you do?"
async with chenna_reddy() as chenna:
    print("🙋", chenna.hello.serverInfo.name, "says:", chenna.hello.instructions, "\n")
    for tool in (await chenna.list_tools()).tools:
        print(f"🔧 {tool.name}: {tool.description.splitlines()[0]}")

✏️ **Your turn:** how many tools does he offer? Type **one number**. That list is what the AI receives. It never sees his code, only the names, descriptions and inputs.

Now the **same goal**, the **same agent**, plus his tools. Watch the plan change by itself.

In [ ]:
async with chenna_reddy() as chenna:
    await run_agent(COMPLAINT, chenna=chenna, name="2 · with Chenna Reddy")

In [ ]:
# 📱 The WhatsApp drafts the agent prepared. Click to open WhatsApp with the message ready; a human presses Send.
from IPython.display import HTML, display
for d in OUTBOX[-2:]:
    display(HTML(f'<p>📱 To <b>{d["to"]}</b>: {d["message"]}<br><a href="{d["link"]}" target="_blank">Open in WhatsApp ↗</a></p>'))

**Why a draft and not a real send?** The agent prepares the message; a person presses Send. That's a **guardrail**: an agent that can message people in your name should get a human check first. (For automatic sending, see the n8n workflow in section 5.)

## 4 · Change the world, watch the plan change

A script follows fixed steps. An agent **re-plans** when something surprises it. Run each experiment, read the 🧠 lines, and see where the plan bends. Each one changes a single thing.

In [ ]:
# 🔄 Experiment A: the warden doesn't pick up.
WORLD["warden_answers"] = False
async with chenna_reddy() as chenna:
    await run_agent(COMPLAINT, chenna=chenna, name="A · warden asleep")
WORLD["warden_answers"] = True

In [ ]:
# 🔄 Experiment B: the orange lever is jammed.
async with chenna_reddy(STUCK_LEVERS="orange") as chenna:
    await run_agent(COMPLAINT, chenna=chenna, name="B · orange jammed")

In [ ]:
# 🔄 Experiment C: the plumber can only come tomorrow.
WORLD["plumber_comes"] = "tomorrow at 9 am"
async with chenna_reddy() as chenna:
    await run_agent(COMPLAINT, chenna=chenna, name="C · plumber tomorrow")
WORLD["plumber_comes"] = "12:45 am"

In [ ]:
# 🔄 Experiment D: a different leak, in B-block.
B_LEAK = "Water leaking from the bathroom ceiling on the B-block 1st floor, near the last cubicle."
async with chenna_reddy(LEAK_AT="B-1") as chenna:
    await run_agent(B_LEAK, chenna=chenna, name="D · B-block leak")

In [ ]:
# 🧾 Every plan, side by side. Same agent, same code: only the world changed.
for name, steps in RUNS.items():
    print(f"{name:24} → {' → '.join(steps)}")

✏️ **Your turn:** which experiment changed the plan the **most**? Type **A, B, C or D**.

✏️ **Try your own:** change one thing (`STUCK_LEVERS="orange,yellow"`, `MAX_STEPS = 4`, or a complaint in Telugu) and run it again. Watch for the **⛔ harness limit** and the **⚠️ red-lever** guardrail.

## 5 · The same agent without code: n8n

n8n builds the same loop with boxes and arrows: **a trigger** (a WhatsApp message arrives) → **an AI Agent node** (Gemini + tools) → **actions** (reply on WhatsApp, write to a Google Sheet). It can even plug in **Mr. Chenna Reddy** through its *MCP Client* node: the same server as above, unchanged.

The ready-made workflows and a step-by-step setup guide are on the course page (Week 7). Your instructor demos them live.

## 6 · Make it yours (your capstone)

Your capstone's "it takes an action" step is exactly this pattern:
1. Write **one tool** that does something real for your project (save to a sheet, send a reminder, look something up).
2. Add it to `LOCAL_TOOLS`, change `SYSTEM` to your project's goal and rules, and call `run_agent` with a real request.
3. Add **one guardrail**: what must it never do without asking?

---
**Next week:** responsible AI, and **demo day**. Bring your project. 🎤